# Tea Shop Sales Analysis & Predictive Modeling
**Comprehensive Practical Notebook — Exploratory Data Analysis, Statistical Profiling, Menu Diagnostics & OLS Sales Forecasting**

* **Course:** Predictive Analytics Lab
* **Dataset Scope:** 20 Transaction Entries (October 1 – October 10, 2026)
* **Product Portfolio:** 4 Artisanal Tea Varieties (*Iced Peach Tea, Masala Chai, Ginger Tea, Green Tea*)
* **Forecast Horizon:** October 11 – October 15, 2026 (Days 11–15)
* **Tools:** Python (`pandas`, `numpy`, `matplotlib`)

## 1. Environment Setup, Data Ingestion & Integrity Audit
We load the required libraries, construct the 20-row transactional dataset (`tea_sales.csv`), dynamically engineer the `Revenue` feature (`Cups_Sold * Price`), and run a schema and data integrity verification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configure plot aesthetics
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# Step 1: Create the 20-row dataset (10 days, 2 products sold per day)
data = {
    "Date": [
        "2026-10-01", "2026-10-01", "2026-10-02", "2026-10-02", "2026-10-03",
        "2026-10-03", "2026-10-04", "2026-10-04", "2026-10-05", "2026-10-05",
        "2026-10-06", "2026-10-06", "2026-10-07", "2026-10-07", "2026-10-08",
        "2026-10-08", "2026-10-09", "2026-10-09", "2026-10-10", "2026-10-10"
    ],
    "Product": [
        "Masala Chai", "Ginger Tea", "Green Tea", "Masala Chai", "Iced Peach Tea",
        "Ginger Tea", "Masala Chai", "Green Tea", "Iced Peach Tea", "Masala Chai",
        "Ginger Tea", "Iced Peach Tea", "Green Tea", "Masala Chai", "Ginger Tea",
        "Iced Peach Tea", "Masala Chai", "Green Tea", "Iced Peach Tea", "Ginger Tea"
    ],
    "Cups_Sold": [
        30, 25, 20, 28, 35, 22, 40, 18, 38, 32,
        27, 42, 21, 35, 30, 45, 38, 24, 50, 33
    ],
    "Price": [
        180, 160, 120, 180, 200, 160, 180, 120, 200, 180,
        160, 200, 120, 180, 160, 200, 180, 120, 200, 160
    ]
}

df = pd.DataFrame(data)
df.to_csv("tea_sales.csv", index=False)

# Step 2: Feature engineering (Revenue & Share of Total %)
df["Revenue"] = df["Cups_Sold"] * df["Price"]
df["Share_of_Total_%"] = (df["Revenue"] / df["Revenue"].sum() * 100).round(2)

# Data Integrity Audit
print("=== DATA INTEGRITY & SCHEMA AUDIT ===")
print(f"Shape           : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Unique Dates    : {df['Date'].nunique()} ({df['Date'].min()} to {df['Date'].max()})")
print(f"Unique Products : {df['Product'].nunique()} -> {list(df['Product'].unique())}")
print(f"Missing Values  : {df.isnull().sum().sum()}")
print(f"Arithmetic Check: {(df['Revenue'] == df['Cups_Sold'] * df['Price']).all()}")

df

## 2. Multi-Dimensional Statistical Summary Matrix
We compute parametric (mean, standard deviation, coefficient of variation) and non-parametric (median, Q1, Q3, IQR, range) distribution metrics across both **Daily Aggregated Totals ($n = 10$)** and **Individual Transaction Entries ($n = 20$)**.

In [ ]:
daily_agg = df.groupby("Date").agg(
    Daily_Revenue=("Revenue", "sum"),
    Daily_Cups=("Cups_Sold", "sum")
).reset_index()

def compute_dist_metrics(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    mean_val = series.mean()
    std_val = series.std(ddof=1)
    return {
        "Sample_Size_n": int(series.count()),
        "Cumulative_Sum": round(float(series.sum()), 2),
        "Mean": round(float(mean_val), 2),
        "Std_Dev_s": round(float(std_val), 2),
        "Min": round(float(series.min()), 2),
        "Q1_25th": round(float(q1), 2),
        "Median_Q2": round(float(series.median()), 2),
        "Q3_75th": round(float(q3), 2),
        "IQR": round(float(q3 - q1), 2),
        "Max": round(float(series.max()), 2),
        "Range": round(float(series.max() - series.min()), 2),
        "CV_%": round(float(std_val / mean_val * 100), 2)
    }

summary_matrix = pd.DataFrame({
    "Daily_Revenue_Rs": compute_dist_metrics(daily_agg["Daily_Revenue"]),
    "Daily_Cups_Sold": compute_dist_metrics(daily_agg["Daily_Cups"]),
    "Entry_Revenue_Rs": compute_dist_metrics(df["Revenue"]),
    "Entry_Cups_Sold": compute_dist_metrics(df["Cups_Sold"])
})

weighted_avg_price = df["Revenue"].sum() / df["Cups_Sold"].sum()
print(f"Weighted Average Realized Price: Rs {weighted_avg_price:.2f} / cup\n")
summary_matrix

## 3. Exploratory Analysis Findings (Q1 – Q4)
* **Q1 — Total Financial & Volume Performance:** Cumulative gross revenue and cups dispensed over 10 days.
* **Q2 — Product Leadership & Pareto Concentration:** Revenue and volume share across the four tea varieties.
* **Q3 — Daily Trajectory & Peak/Trough Identification:** Day-by-day revenue progression, 3-day trailing simple moving average ($SMA_3$), and variance against the daily mean.
* **Q4 — Item-Level Statistical Dispersion:** Mean, median, standard deviation, and coefficient of variation per active entry.

In [ ]:
# Q1: Aggregate Earnings & Volume
total_revenue = df["Revenue"].sum()
total_cups = df["Cups_Sold"].sum()
print("=== Q1: TOTAL FINANCIAL & VOLUME PERFORMANCE ===")
print(f"Total Gross Revenue : Rs {total_revenue:,}")
print(f"Total Cups Sold     : {total_cups} cups\n")

# Q2: Product-Wise Hierarchy & Cumulative Pareto Share
product_summary = df.groupby("Product").agg(
    Unit_Price_Rs=("Price", "first"),
    Active_Days=("Date", "count"),
    Total_Cups=("Cups_Sold", "sum"),
    Gross_Revenue_Rs=("Revenue", "sum")
).sort_values("Gross_Revenue_Rs", ascending=False)

product_summary["Volume_Share_%"] = (product_summary["Total_Cups"] / total_cups * 100).round(2)
product_summary["Revenue_Share_%"] = (product_summary["Gross_Revenue_Rs"] / total_revenue * 100).round(2)
product_summary["Cum_Rev_Share_%"] = product_summary["Revenue_Share_%"].cumsum().round(2)

print("=== Q2: PRODUCT-WISE PERFORMANCE HIERARCHY ===")
print(product_summary)
print(f"\nTop Commercial Leader: {product_summary.index[0]} (Rs {product_summary['Gross_Revenue_Rs'].iloc[0]:,})")

# Q4: Item-Level Statistical Distribution
item_stats = df.groupby("Product")["Cups_Sold"].agg(
    Entries="count",
    Mean_Cups="mean",
    Median_Cups="median",
    Std_Dev="std",
    Min="min",
    Max="max"
).round(2)
item_stats["CV_%"] = (item_stats["Std_Dev"] / item_stats["Mean_Cups"] * 100).round(2)
item_stats = item_stats.loc[product_summary.index]

print("\n=== Q4: ITEM-LEVEL STATISTICAL DISPERSION ===")
print(item_stats)

## 4. Daily Chronological Audit, Menu Pairing & Period-Over-Period Diagnostics
We examine each operational day's active menu pair, 3-day trailing Simple Moving Average ($SMA_3$), percentage deviation from the ₹11,042 daily mean, and compare **First Half (Oct 1–5)** against **Second Half (Oct 6–10)** performance.

In [ ]:
# Q3: Daily Chronological Audit with Featured Menu Pair & 3-Day SMA
daily_audit = df.groupby("Date").agg(
    Featured_Menu_Pair=("Product", lambda s: " + ".join(s)),
    Daily_Cups=("Cups_Sold", "sum"),
    Actual_Rev_Rs=("Revenue", "sum")
).reset_index()

daily_audit.insert(1, "Day_Index_t", np.arange(1, len(daily_audit) + 1))
daily_audit["SMA_3_Rs"] = daily_audit["Actual_Rev_Rs"].rolling(window=3).mean().round(2)
mean_daily_rev = daily_audit["Actual_Rev_Rs"].mean()
daily_audit["Vs_Mean_%"] = ((daily_audit["Actual_Rev_Rs"] - mean_daily_rev) / mean_daily_rev * 100).round(2)

print("=== Q3: DAILY CHRONOLOGICAL AUDIT ===")
print(daily_audit.to_string(index=False))
print(f"\nPeak Day  : {daily_audit.loc[daily_audit['Actual_Rev_Rs'].idxmax(), 'Date']} -> Rs {daily_audit['Actual_Rev_Rs'].max():,}")
print(f"Trough Day: {daily_audit.loc[daily_audit['Actual_Rev_Rs'].idxmin(), 'Date']} -> Rs {daily_audit['Actual_Rev_Rs'].min():,}")

# Period-over-Period Comparison: First Half (Days 1-5) vs Second Half (Days 6-10)
h1 = daily_audit.iloc[:5]
h2 = daily_audit.iloc[5:]

period_comp = pd.DataFrame({
    "Metric": ["Cumulative Gross Revenue (Rs)", "Cumulative Cups Sold", "Mean Daily Gross Revenue (Rs)", "Mean Daily Cup Volume"],
    "First_Half_Oct_1_5": [h1["Actual_Rev_Rs"].sum(), h1["Daily_Cups"].sum(), h1["Actual_Rev_Rs"].mean(), h1["Daily_Cups"].mean()],
    "Second_Half_Oct_6_10": [h2["Actual_Rev_Rs"].sum(), h2["Daily_Cups"].sum(), h2["Actual_Rev_Rs"].mean(), h2["Daily_Cups"].mean()]
})
period_comp["Net_Change"] = period_comp["Second_Half_Oct_6_10"] - period_comp["First_Half_Oct_1_5"]
period_comp["Growth_%"] = (period_comp["Net_Change"] / period_comp["First_Half_Oct_1_5"] * 100).round(2)

print("\n=== PERIOD-OVER-PERIOD ACCELERATION (OCT 1-5 VS OCT 6-10) ===")
print(period_comp.to_string(index=False))

## 5. Predictive Analytics — OLS Linear Trend Regression & Multi-Day Forecasting (Q5 & Q6)
* **Q5 — Top-Line Daily Revenue Regression ($t = 1 \dots 10$):** Fits $\hat{y}_t = \beta_1 t + \beta_0$, evaluates $R^2$, MAE, RMSE, MAPE, in-sample residuals, and generates out-of-sample forecasts for **Days 11–15 (Oct 11–15, 2026)** alongside a 95% prediction interval.
* **Q6 — Product-Specific Next-Appearance Volume Forecasts:** Fits individual OLS trend models to each beverage's chronological appearance sequence and recommends a +15% kitchen prep par buffer.

In [ ]:
# Q5: OLS Linear Regression on Daily Revenue (t = 1..10)
X = daily_audit["Day_Index_t"].values
y = daily_audit["Actual_Rev_Rs"].values

slope, intercept = np.polyfit(X, y, 1)
y_pred = slope * X + intercept
residuals = y - y_pred

# Goodness-of-Fit & Accuracy Metrics
n = len(y)
sse = np.sum(residuals ** 2)
sst = np.sum((y - np.mean(y)) ** 2)
r_squared = 1 - (sse / sst)
mae = np.mean(np.abs(residuals))
rmse = np.sqrt(np.mean(residuals ** 2))
mape = np.mean(np.abs(residuals / y)) * 100

print("=== Q5: OLS LINEAR TREND MODEL DIAGNOSTICS ===")
print(f"Trend Equation : Revenue(t) = {slope:.2f} * t + {intercept:.2f}")
print(f"R-squared      : {r_squared:.4f} ({r_squared*100:.2f}% variance explained)")
print(f"MAE            : Rs {mae:,.2f}")
print(f"RMSE           : Rs {rmse:,.2f}")
print(f"MAPE           : {mape:.2f}%\n")

# In-Sample Residual Table
residual_df = pd.DataFrame({
    "Date": daily_audit["Date"],
    "Day_t": X,
    "Actual_Rev_Rs": y,
    "OLS_Predicted_Rs": y_pred.round(2),
    "Residual_e_Rs": residuals.round(2),
    "Abs_Error_%": (np.abs(residuals / y) * 100).round(2)
})
print("=== IN-SAMPLE RESIDUAL DIAGNOSTICS ===")
print(residual_df.to_string(index=False))

# 95% Prediction Interval for Day 11 (t_crit = 2.306 for df = 8)
s_err = np.sqrt(sse / (n - 2))
x_mean = np.mean(X)
sxx = np.sum((X - x_mean) ** 2)
t_crit = 2.306004

future_days = np.array([11, 12, 13, 14, 15])
future_dates = [f"2026-10-{d}" for d in future_days]
future_preds = slope * future_days + intercept
pi_margin = t_crit * s_err * np.sqrt(1 + (1 / n) + ((future_days - x_mean) ** 2) / sxx)

forecast_df = pd.DataFrame({
    "Forecast_Date": future_dates,
    "Day_Index_t": future_days,
    "OLS_Point_Forecast_Rs": future_preds.round(2),
    "PI_95_Lower_Rs": (future_preds - pi_margin).round(2),
    "PI_95_Upper_Rs": (future_preds + pi_margin).round(2),
    "Est_Cups_at_174_44": (future_preds / weighted_avg_price).round(1)
})

print("\n=== MULTI-DAY HORIZON FORECAST (OCT 11 – OCT 15, 2026) ===")
print(forecast_df.to_string(index=False))

# Q6: Product-Specific Next-Appearance Demand Forecasts
prod_forecasts = []
for prod in ["Iced Peach Tea", "Masala Chai", "Ginger Tea", "Green Tea"]:
    sub = df[df["Product"] == prod].reset_index(drop=True)
    idx = np.arange(1, len(sub) + 1)
    vols = sub["Cups_Sold"].values
    m, b = np.polyfit(idx, vols, 1)
    next_vol = m * (len(sub) + 1) + b
    unit_price = sub["Price"].iloc[0]
    prod_forecasts.append({
        "Product_Name": prod,
        "Historical_Entries": len(sub),
        "Slope_Cups_Per_Entry": round(float(m), 2),
        "Next_Entry_Forecast_Cups": round(float(next_vol), 1),
        "Recommended_Prep_Par_+15%": int(np.ceil(next_vol * 1.15)),
        "Expected_Revenue_Rs": round(float(next_vol * unit_price), 2)
    })

prod_forecast_df = pd.DataFrame(prod_forecasts)
print("\n=== Q6: PRODUCT-SPECIFIC NEXT-APPEARANCE DEMAND FORECASTS ===")
print(prod_forecast_df.to_string(index=False))

## 6. Visual Analytics Dashboard
We generate a comprehensive 4-panel visual analytics dashboard:
1. **Gross Revenue by Tea Product** (Bar Chart)
2. **Share of Total Cups Sold** (Pie Chart)
3. **Daily Revenue Trajectory, 3-Day SMA & OLS Trend Forecast** (Line & Scatter Chart)
4. **In-Sample OLS Residuals by Menu Pairing** (Diagnostic Bar Chart)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Panel 1: Gross Revenue by Tea Product
colors_bar = ["#059669", "#0284c7", "#d97706", "#65a30d"]
bars = axes[0, 0].bar(product_summary.index, product_summary["Gross_Revenue_Rs"], color=colors_bar)
axes[0, 0].set_title("1. Gross Revenue by Tea Product (Oct 1–10)", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Gross Revenue (Rs)")
for bar in bars:
    h = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, h + 700, f"Rs {h:,.0f}", ha="center", fontsize=9, fontweight="bold")

# Panel 2: Share of Cups Sold
axes[0, 1].pie(
    product_summary["Total_Cups"],
    labels=product_summary.index,
    autopct="%1.2f%%",
    startangle=140,
    colors=colors_bar,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
axes[0, 1].set_title("2. Share of Total Cups Sold (633 Cups)", fontsize=12, fontweight="bold")

# Panel 3: Daily Revenue Trajectory, 3-Day SMA & OLS Forecast
axes[1, 0].plot(X, y, marker="o", color="#0284c7", linewidth=2, label="Actual Daily Revenue")
axes[1, 0].plot(X, daily_audit["SMA_3_Rs"], linestyle=":", marker="s", color="#059669", label="3-Day Trailing SMA")
t_ext = np.arange(1, 16)
axes[1, 0].plot(t_ext, slope * t_ext + intercept, linestyle="--", color="#d97706", label="OLS Trend Line (+Rs 551.76/day)")
axes[1, 0].scatter([11], [future_preds[0]], color="#dc2626", s=90, zorder=5, label=f"Day 11 Forecast (Rs {future_preds[0]:,.0f})")
axes[1, 0].fill_between(future_days, future_preds - pi_margin, future_preds + pi_margin, color="#dc2626", alpha=0.12, label="95% Prediction Interval")
axes[1, 0].set_title("3. Daily Revenue Trajectory & 5-Day OLS Forecast", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Operational Day Index (Oct 1 = Day 1)")
axes[1, 0].set_ylabel("Revenue (Rs)")
axes[1, 0].set_xticks(range(1, 16))
axes[1, 0].legend(fontsize=8, loc="upper left")

# Panel 4: In-Sample OLS Residuals
res_colors = ["#059669" if r >= 0 else "#dc2626" for r in residuals]
axes[1, 1].bar(X, residuals, color=res_colors, alpha=0.85)
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set_title("4. In-Sample OLS Residuals (Menu Pairing Effect)", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Operational Day Index (t)")
axes[1, 1].set_ylabel("Residual Error (Rs)")
axes[1, 1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig("tea_shop_analytics_dashboard.png", dpi=150)
plt.show()